# Agentic Workflows as Probabilistic Programs

LLMs are increasingly being used as agents when it comes to solving problems and tasks. These scenarios involve single or multiple LLMs interacting with the environment via tools and gathering information in order to accomplish a task or to find answer to a complex query. There are several workflows that people have come up with based on specific tasks and tools available for LLMs to use. Some notable examples include Self-Refine, Reflexion, ReAct, Magentic-One etc.

Since language models are generative, we envision these workflows to be probabilistic programs. With this perspective, we aim to separate the "what" from "how", the declarative specification of the task from how it is achieved. Now, which components of these workflows belong in "what" and which components belong in "how" is unfortunately subjective and the right choice depends on which separation opens the most uses. One perspective where everything other than the query and the final answer belongs in "how" is described in this [overleaf document](https://www.overleaf.com/read/vkkvczrnnkwb#e0fe1e)

In this note, we focus on a different perspective where a few more components than the query and the final answer make it to "what". This is primarily done to allow reasoning about a wider set of components of these agentic workflows.

## ReAct

A typical ReAct pipeline sends an LLM two prompts 

1) A system prompt that tells you what sequence of steps to follow 
2) A task prompt describing the query at hand. Upon receiving these prompts, the LLM goes through a loop of outputting thoughts and code blocks which are executed, appended to the prompt and the loop continues until final_answer is reached.

Algorithmically, it looks something like below:

```python
system_prompt = …
task = …
is_final = False
memory = task + system_prompt
while not is_final:
	thought, action = sample(LLM(.. | memory))
	is_final = check(action)
	if is_final:
		answer = extract_answer(action)
	else:
		observation = exec(action)
		memory += observation
return answer
```

The above code is currently implemented as CodeAgent in smolagents library.



This [notebook](playground/fun.ipynb) uses CodeAgent in smolagent library to carry out a ReAct loop on the first 20 examples of GAIA benchmark. It shows 20% accuracy which varies significantly from run to run and uses on average 229,556.85 tokens per question which is a lot since the limit of context window for OpenAI API is 128,000.

## Token discrepancies

It can be confusing in smolagents that how are the statistics about input and output tokens are being generated. Atleast for OpenAI model, these token statistics printed by the library are cumulative input and output tokens totaled over all the API calls made for a query till a particular point. This differs from the length of the context passed to the single call to API.

Also, note that OpenAI APIs perform [**prompt caching**](https://developers.openai.com/api/docs/guides/prompt-caching) which can affect how the price of the tokens differ.

## Baselines

Before we move on, let's see how different LLMs perform as backbones for ReAct. 

- The following list does not include deepseek-ai/DeepSeek-R1 because it is being deprecated and not available for serverless use on TogetherAI.
- The Qwen models are being used with `{"enable_thinking": False}` as a parameter.

The two datasets being used are 
- GAIA (General AI Assistant) - 165 questions
- Fermi (Guesstimate problems) - 125 questions

In [1]:
import pickle
from pathlib import Path

import pandas as pd
from IPython.display import display

BASELINE_DIR = Path("baseline")

# Read from the per-example .pkl cache dirs rather than the .jsonl logs: evaluate_agent
# writes pickle_dir/{i}.pkl once per example index, overwritten in place on recompute, so
# each file always reflects that example's single latest result -- no de-duplication needed,
# unlike the .jsonl logs which are append-only across every run/restart across sessions.
MODEL_DIRS = {
    "gpt-4o": BASELINE_DIR / "naive_react_gpt-4o_False",
    "gpt-5.4-mini": BASELINE_DIR / "naive_react_gpt-5.4-mini_False",
    "Qwen3.7-Plus": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.7-Plus_False",
    "Qwen3.5-9B": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.5-9B_False",
}

# Fermi (RealFP) naive ReAct baselines -- same models, different dataset/scorer (see
# baseline/fermi_setup.py). GAIA's is_correct is an exact-match boolean; Fermi's is a
# continuous 0-1 order-of-magnitude score from fermi_scorer. summarize_results below handles
# both uniformly via float(), since True/False cast cleanly to 1.0/0.0.
FERMI_MODEL_DIRS = {
    "gpt-4o": BASELINE_DIR / "naive_fermi_gpt-4o",
    "gpt-5.4-mini": BASELINE_DIR / "naive_fermi_gpt-5.4-mini",
    "Qwen3.7-Plus": BASELINE_DIR / "naive_fermi_Qwen_Qwen3.7-Plus",
    "Qwen3.5-9B": BASELINE_DIR / "naive_fermi_Qwen_Qwen3.5-9B",
}


def summarize_results(pkl_dir: Path):
    records = []
    for pkl_file in sorted(pkl_dir.glob("*.pkl"), key=lambda p: int(p.stem)):
        with open(pkl_file, "rb") as f:
            row = pickle.load(f)
        token_counts = row.get("token_counts") or {}
        records.append({
            "is_correct": float(row.get("is_correct", False) or 0.0),
            "num_steps": row.get("num_steps", 0),
            "input_tokens": token_counts.get("input_tokens", 0),
            "output_tokens": token_counts.get("output_tokens", 0),
            "total_tokens": token_counts.get("total_tokens", 0),
            "error": row.get("error"),
        })

    n = len(records)
    if n == 0:
        return None
    correct = sum(r["is_correct"] for r in records)
    return {
        "n": n,
        "correct": correct,
        "accuracy": correct / n,
        "avg_steps": sum(r["num_steps"] for r in records) / n,
        "avg_input_tokens": sum(r["input_tokens"] for r in records) / n,
        "avg_output_tokens": sum(r["output_tokens"] for r in records) / n,
        "avg_tokens": sum(r["total_tokens"] for r in records) / n,
        "errors": sum(1 for r in records if r["error"]),
    }


def summarize_rows(model_dirs: dict):
    rows = []
    for model_name, pkl_dir in model_dirs.items():
        if not pkl_dir.exists():
            rows.append({"Model": model_name, "Accuracy": "missing dir", "Questions": 0, "Avg Steps": "-", "Avg Input Tokens": "-", "Avg Output Tokens": "-", "Avg Tokens": "-", "Errors": "-"})
            continue
        stats = summarize_results(pkl_dir)
        if stats is None:
            rows.append({"Model": model_name, "Accuracy": "no data", "Questions": 0, "Avg Steps": "-", "Avg Input Tokens": "-", "Avg Output Tokens": "-", "Avg Tokens": "-", "Errors": "-"})
            continue
        # correct is a whole number for GAIA (sum of 1.0/0.0) but fractional for Fermi (sum of
        # continuous scores) -- show 0 decimals for the former, 1 for the latter.
        correct_display = f"{stats['correct']:.0f}" if stats["correct"] == round(stats["correct"]) else f"{stats['correct']:.1f}"
        rows.append({
            "Model": model_name,
            "Accuracy": f"{correct_display}/{stats['n']} = {stats['accuracy']:.1%}",
            "Questions": stats["n"],
            "Avg Steps": f"{stats['avg_steps']:.1f}",
            "Avg Input Tokens": f"{stats['avg_input_tokens']:,.0f}",
            "Avg Output Tokens": f"{stats['avg_output_tokens']:,.0f}",
            "Avg Tokens": f"{stats['avg_tokens']:,.0f}",
            "Errors": stats["errors"],
        })
    return rows


print("=== GAIA ===")
display(pd.DataFrame(summarize_rows(MODEL_DIRS)))

print("=== Fermi ===")
display(pd.DataFrame(summarize_rows(FERMI_MODEL_DIRS)))

=== GAIA ===


,Model,Accuracy,Questions,Avg Steps,Avg Input Tokens,Avg Output Tokens,Avg Tokens,Errors
0,gpt-4o,43/165 = 26.1%,165,10.8,"134,996","1,609","136,605",0
1,gpt-5.4-mini,54/165 = 32.7%,165,5.4,"36,684",607,"37,291",0
2,Qwen3.7-Plus,103/165 = 62.4%,165,19.5,"452,400","3,028","455,429",1
3,Qwen3.5-9B,83/165 = 50.3%,165,18.9,"463,766","4,031","467,797",0


=== Fermi ===


,Model,Accuracy,Questions,Avg Steps,Avg Input Tokens,Avg Output Tokens,Avg Tokens,Errors
0,gpt-4o,71.5/125 = 57.2%,125,5.2,"40,780",666,"41,446",0
1,gpt-5.4-mini,55.3/125 = 44.2%,125,2.2,"6,495",93,"6,588",0
2,Qwen3.7-Plus,76.6/125 = 61.3%,125,16.1,"296,868","2,674","299,542",0
3,Qwen3.5-9B,69.9/125 = 55.9%,125,11.5,"173,161","2,861","176,022",0


Based on the above results, it does seem that open-source models use more tokens and more steps but are performing comparable or even better than closed source models. So maybe we should simply shift to open-source models.

## Markov ReAct

Another dimension that we wanted to test was how independent is each iteration of ReAct in comparison to the previous ones. Can the previous iterations be completely pruned and how much would this impact the performance?

In [2]:
from IPython.display import display

MARKOV_DIR = Path("markovReAct")

# Reuses summarize_results and MODEL_DIRS/FERMI_MODEL_DIRS (naive baselines) defined in the
# Baselines cell above.
CONFIG_DIRS = {
    "gpt-4o": {
        "naive": MODEL_DIRS["gpt-4o"],
        "w1": MARKOV_DIR / "markov_react_gpt-4o_w1",
        "w3": MARKOV_DIR / "markov_react_gpt-4o_w3",
        "w5": MARKOV_DIR / "markov_react_gpt-4o_w5",
    },
    "gpt-5.4-mini": {
        "naive": MODEL_DIRS["gpt-5.4-mini"],
        "w1": MARKOV_DIR / "markov_react_gpt-5.4-mini_w1",
        "w3": MARKOV_DIR / "markov_react_gpt-5.4-mini_w3",
        "w5": MARKOV_DIR / "markov_react_gpt-5.4-mini_w5",
    },
    "Qwen3.7-Plus": {
        "naive": MODEL_DIRS["Qwen3.7-Plus"],
        "w1": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w1",
        "w3": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w3",
        "w5": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w5",
    },
    "Qwen3.5-9B": {
        "naive": MODEL_DIRS["Qwen3.5-9B"],
        "w1": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w1",
        "w3": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w3",
        "w5": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w5",
    },
}

# Fermi MarkovReAct -- same window sizes, same models, different dataset/scorer.
FERMI_CONFIG_DIRS = {
    "gpt-4o": {
        "naive": FERMI_MODEL_DIRS["gpt-4o"],
        "w1": MARKOV_DIR / "markov_fermi_gpt-4o_w1",
        "w3": MARKOV_DIR / "markov_fermi_gpt-4o_w3",
        "w5": MARKOV_DIR / "markov_fermi_gpt-4o_w5",
    },
    "gpt-5.4-mini": {
        "naive": FERMI_MODEL_DIRS["gpt-5.4-mini"],
        "w1": MARKOV_DIR / "markov_fermi_gpt-5.4-mini_w1",
        "w3": MARKOV_DIR / "markov_fermi_gpt-5.4-mini_w3",
        "w5": MARKOV_DIR / "markov_fermi_gpt-5.4-mini_w5",
    },
    "Qwen3.7-Plus": {
        "naive": FERMI_MODEL_DIRS["Qwen3.7-Plus"],
        "w1": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.7-Plus_w1",
        "w3": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.7-Plus_w3",
        "w5": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.7-Plus_w5",
    },
    "Qwen3.5-9B": {
        "naive": FERMI_MODEL_DIRS["Qwen3.5-9B"],
        "w1": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.5-9B_w1",
        "w3": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.5-9B_w3",
        "w5": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.5-9B_w5",
    },
}
CONFIG_ORDER = ["naive", "w1", "w3", "w5"]


def build_results_df(config_dirs: dict) -> pd.DataFrame:
    rows = []
    for model_name, configs in config_dirs.items():
        for config_name, pkl_dir in configs.items():
            stats = summarize_results(pkl_dir) if pkl_dir.exists() else None
            if stats is None:
                rows.append({"Model": model_name, "Config": config_name, "Accuracy": None, "Avg Steps": None, "Tokens (in/out/total)": None})
                continue
            rows.append({
                "Model": model_name,
                "Config": config_name,
                "Accuracy": f"{stats['accuracy']:.1%}",
                "Avg Steps": round(stats["avg_steps"], 1),
                "Tokens (in/out/total)": f"{stats['avg_input_tokens']:,.0f} / {stats['avg_output_tokens']:,.0f} / {stats['avg_tokens']:,.0f}",
            })
    return pd.DataFrame(rows)


gaia_results_df = build_results_df(CONFIG_DIRS)
fermi_results_df = build_results_df(FERMI_CONFIG_DIRS)

print("##### GAIA #####")
for metric in ["Accuracy", "Avg Steps", "Tokens (in/out/total)"]:
    print(f"=== {metric} ===")
    display(gaia_results_df.pivot(index="Model", columns="Config", values=metric)[CONFIG_ORDER])

print("\n##### Fermi (Avg Score in place of Accuracy) #####")
for metric in ["Accuracy", "Avg Steps", "Tokens (in/out/total)"]:
    print(f"=== {metric} ===")
    display(fermi_results_df.pivot(index="Model", columns="Config", values=metric)[CONFIG_ORDER])

##### GAIA #####
=== Accuracy ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,50.3%,29.1%,33.9%,44.2%
Qwen3.7-Plus,62.4%,29.7%,54.5%,53.9%
gpt-4o,26.1%,28.5%,28.5%,30.9%
gpt-5.4-mini,32.7%,27.9%,33.9%,30.9%


=== Avg Steps ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,18.9,30.0,26.6,24.0
Qwen3.7-Plus,19.5,32.6,24.7,22.4
gpt-4o,10.8,21.0,13.4,11.9
gpt-5.4-mini,5.4,8.6,5.2,5.0


=== Tokens (in/out/total) ===


...

Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,55.9%,54.6%,52.5%,58.5%
Qwen3.7-Plus,61.3%,51.5%,55.1%,55.4%
gpt-4o,57.2%,59.5%,55.6%,57.9%
gpt-5.4-mini,44.2%,42.5%,45.1%,39.2%


=== Avg Steps ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,11.5,21.0,16.8,14.3
Qwen3.7-Plus,16.1,21.7,20.6,17.8
gpt-4o,5.2,4.8,4.2,4.5
gpt-5.4-mini,2.2,2.2,2.2,2.2


=== Tokens (in/out/total) ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,"173,161 / 2,861 / 176,022","86,114 / 2,867 / 88,980","106,039 / 2,978 / 109,017","118,734 / 2,781 / 121,515"
Qwen3.7-Plus,"296,868 / 2,674 / 299,542","90,431 / 2,030 / 92,461","128,061 / 2,310 / 130,371","140,663 / 2,336 / 142,999"
gpt-4o,"40,780 / 666 / 41,446","17,249 / 657 / 17,907","20,246 / 560 / 20,806","24,687 / 612 / 25,299"
gpt-5.4-mini,"6,495 / 93 / 6,588","6,326 / 82 / 6,408","6,348 / 95 / 6,443","6,455 / 98 / 6,553"


The above results are very interesting as they show how the number of steps are more in Markov ReAct but the token counts are low. It turns out the source of low token counts is low input tokens. And in fact, the output tokens are higher in Markov ReAct than naive ReAct.

It is also worth noting that for gpt-4o and gpt-5.4-mini, naiveReAct is not the most accurate version which indicates to possibility of improving performance when the context is cleverly chosen.

## Quantifying Redundancies in Steps

In this [notebook](step_independence/step_independence.ipynb), we go through all the trajectories and ask "gpt-4o-mini" to judge whether the steps were redundant. We got the following results:

In [3]:
import json

STEP_INDEPENDENCE_DIR = Path("step_independence")

# Reuses CONFIG_DIRS/FERMI_CONFIG_DIRS (model -> {naive/w1/w3/w5: pkl_dir}) and CONFIG_ORDER
# defined in the Markov ReAct cell above.

def load_redundant_pairs(model_name: str, config_name: str, prefix: str = "") -> list[dict]:
    key = model_name if config_name == "naive" else f"{model_name}_{config_name}"
    path = STEP_INDEPENDENCE_DIR / f"redundant_pairs_{prefix}{key}.json"
    if not path.exists():
        return []
    with open(path) as f:
        data = json.load(f)
    return data.get("findings", [])


def total_pairs(pkl_dir: Path) -> int:
    """Total number of consecutive step-pairs across every trajectory in pkl_dir."""
    total = 0
    for pkl_file in pkl_dir.glob("*.pkl"):
        with open(pkl_file, "rb") as f:
            traj = pickle.load(f)
        total += max(0, traj.get("num_steps", 0) - 1)
    return total


def build_redundancy_df(config_dirs: dict, prefix: str = "") -> pd.DataFrame:
    rows = []
    for model_name, configs in config_dirs.items():
        for config_name, pkl_dir in configs.items():
            n_redundant = len(load_redundant_pairs(model_name, config_name, prefix))
            n_total = total_pairs(pkl_dir) if pkl_dir.exists() else 0
            rows.append({
                "Model": model_name,
                "Config": config_name,
                "Redundant Pairs": n_redundant,
                "% Redundant": round(100 * n_redundant / n_total, 1) if n_total else None,
            })
    return pd.DataFrame(rows)


redundancy_df = build_redundancy_df(CONFIG_DIRS)
fermi_redundancy_df = build_redundancy_df(FERMI_CONFIG_DIRS, prefix="fermi_")

print("##### GAIA #####")
for metric in ["Redundant Pairs", "% Redundant"]:
    print(f"=== {metric} ===")
    display(redundancy_df.pivot(index="Model", columns="Config", values=metric)[CONFIG_ORDER])

print("\n##### Fermi #####")
for metric in ["Redundant Pairs", "% Redundant"]:
    print(f"=== {metric} ===")
    display(fermi_redundancy_df.pivot(index="Model", columns="Config", values=metric)[CONFIG_ORDER])

##### GAIA #####
=== Redundant Pairs ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,632,1350,1096,908
Qwen3.7-Plus,710,2053,1124,950
gpt-4o,381,796,470,393
gpt-5.4-mini,118,222,96,93


=== % Redundant ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,21.4,28.3,26.0,23.9
Qwen3.7-Plus,23.3,39.3,28.7,27.0
gpt-4o,23.5,24.1,22.9,21.9
gpt-5.4-mini,16.1,17.7,14.0,14.0



##### Fermi #####
=== Redundant Pairs ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,228,995,510,363
Qwen3.7-Plus,324,833,568,426
gpt-4o,130,89,93,78
gpt-5.4-mini,32,31,36,32


=== % Redundant ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,17.3,39.7,25.8,21.8
Qwen3.7-Plus,17.2,32.1,23.1,20.3
gpt-4o,25.0,18.5,23.1,17.9
gpt-5.4-mini,20.9,20.1,24.0,21.3


Based on the numbers above, it seems that for Qwen models, as you prune the context, the relative redundancy increases. The same does not hold for gpt models, apparently window context 3 and 5 have lower relative redundancies compared to full context.

It also seems that there might be some correlation between accuracy and redundancy:

In [4]:
# Reuses CONFIG_DIRS/FERMI_CONFIG_DIRS, summarize_results, and redundancy_df/fermi_redundancy_df
# from the cells above.

def build_corr_df(config_dirs: dict, redundancy_df: pd.DataFrame) -> pd.DataFrame:
    accuracy_rows = []
    for model_name, configs in config_dirs.items():
        for config_name, pkl_dir in configs.items():
            stats = summarize_results(pkl_dir) if pkl_dir.exists() else None
            accuracy_rows.append({
                "Model": model_name,
                "Config": config_name,
                "Accuracy": stats["accuracy"] if stats else None,
            })
    accuracy_df = pd.DataFrame(accuracy_rows)
    return redundancy_df.merge(accuracy_df, on=["Model", "Config"])


gaia_corr_df = build_corr_df(CONFIG_DIRS, redundancy_df)
fermi_corr_df = build_corr_df(FERMI_CONFIG_DIRS, fermi_redundancy_df)

print("GAIA -- per-model correlation (accuracy vs. % redundant), n=4 configs (naive, w1, w3, w5) each:")
for model_name, group in gaia_corr_df.groupby("Model"):
    corr = group["Accuracy"].corr(group["% Redundant"])
    print(f"  {model_name}: r = {corr:.3f}")

print("\nFermi -- per-model correlation (avg score vs. % redundant), n=4 configs (naive, w1, w3, w5) each:")
for model_name, group in fermi_corr_df.groupby("Model"):
    corr = group["Accuracy"].corr(group["% Redundant"])
    print(f"  {model_name}: r = {corr:.3f}")

GAIA -- per-model correlation (accuracy vs. % redundant), n=4 configs (naive, w1, w3, w5) each:
  Qwen3.5-9B: r = -0.988
  Qwen3.7-Plus: r = -0.992
  gpt-4o: r = -0.696
  gpt-5.4-mini: r = -0.706

Fermi -- per-model correlation (avg score vs. % redundant), n=4 configs (naive, w1, w3, w5) each:
  Qwen3.5-9B: r = -0.372
  Qwen3.7-Plus: r = -0.900
  gpt-4o: r = -0.676
  gpt-5.4-mini: r = 0.460


As you can see above, apparently there is a strong negative correlation for GAIA between accuracy and the percentage of redundant step-pairs, suggesting that models with fewer redundant steps tend to perform better on the task.

Results for Fermi are weird I don't get it.

However, the correlation above is computed at the level of (model, config) aggregates (n=4 points per model). Let's check whether the same relationship holds at the level of individual questions -- for each (model, config), correlate `is_correct` against that question's own fraction of redundant step-pairs:

In [5]:
from collections import Counter

# Reuses CONFIG_DIRS, FERMI_CONFIG_DIRS, CONFIG_ORDER, and load_redundant_pairs from the cells above.

def question_level_stats(model_name: str, config_name: str, pkl_dir: Path, prefix: str = "") -> list[dict]:
    """One row per question in this (model, config): its score (float -- 1.0/0.0 for GAIA's
    exact-match boolean, continuous 0-1 for Fermi), and the fraction of its own consecutive
    step-pairs the judge flagged as redundant."""
    findings = load_redundant_pairs(model_name, config_name, prefix)
    redundant_counts = Counter(f["task_id"] for f in findings)

    rows = []
    for pkl_file in pkl_dir.glob("*.pkl"):
        with open(pkl_file, "rb") as f:
            traj = pickle.load(f)
        task_id = traj.get("task_id", "?")
        n_pairs = max(0, traj.get("num_steps", 0) - 1)
        if n_pairs == 0:
            continue
        rows.append({
            "is_correct": float(traj.get("is_correct", False) or 0.0),
            "fraction_redundant": redundant_counts.get(task_id, 0) / n_pairs,
        })
    return rows


def build_question_corr_df(config_dirs: dict, prefix: str = "") -> pd.DataFrame:
    rows = []
    for model_name, configs in config_dirs.items():
        for config_name, pkl_dir in configs.items():
            if not pkl_dir.exists():
                continue
            qrows = question_level_stats(model_name, config_name, pkl_dir, prefix)
            qdf = pd.DataFrame(qrows)
            corr = qdf["is_correct"].corr(qdf["fraction_redundant"])
            rows.append({"Model": model_name, "Config": config_name, "r": round(corr, 3)})
    return pd.DataFrame(rows)


gaia_question_corr_df = build_question_corr_df(CONFIG_DIRS)
fermi_question_corr_df = build_question_corr_df(FERMI_CONFIG_DIRS, prefix="fermi_")

print("##### GAIA #####")
display(gaia_question_corr_df.pivot(index="Model", columns="Config", values="r")[CONFIG_ORDER])

print("\n##### Fermi #####")
display(fermi_question_corr_df.pivot(index="Model", columns="Config", values="r")[CONFIG_ORDER])

##### GAIA #####


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,-0.085,-0.225,-0.208,-0.328
Qwen3.7-Plus,-0.151,-0.445,-0.308,-0.263
gpt-4o,0.048,-0.055,-0.093,0.015
gpt-5.4-mini,0.001,-0.079,-0.041,0.025



##### Fermi #####


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,-0.148,-0.343,-0.098,-0.230
Qwen3.7-Plus,-0.256,-0.318,-0.158,-0.239
gpt-4o,0.094,0.127,0.191,-0.049
gpt-5.4-mini,-0.152,-0.164,-0.307,-0.206


## Probabilistic ReAct

A typical ReAct workflow struggles with various shortcomings, namely
1. The context-length becomes a bottleneck as all triplets of (thought, action and observation) are added to the context indiscriminately.
    a. And markov ReAct was an attempt to evaluate the impact of truncating the context
2. The uncertainty is never handled explicitly making it difficult for the responses to be reliable
3. There is no modularity, it is essentially a continuous global sequence of thoughts, actions and observations.

Essentially, the first and foremost goal is to introduce structure in ReAct in the form of a probabilistic program. I made two attempts to do so: ProbabilisticAgent and ContextPruningAgent, but they both suffer with issues and I won't be describing them for now (TODO).

For now, we plan to try the following algorithm:

```python
system_prompt = …
task = …
is_final = False
memory = task + system_prompt
while not is_final:
    # Sample an action specification from the language model
    thought, action_spec = sample(LLM(.. | memory))
    # Sample n actions using the action specification
    action = sample(LLM(.. | memory + "fill in the specification"), n)
    results = execute(action[1:n])
    # Use LLM-as-a-judge to pick which step makes the most progress
    feedback = sample(LLM(.. | task + "Look at these {n} pairs of thought action and results and rank them"))
    final_thought, final_action, final_result = select(feedback, thought, action, result)
	is_final = check(final_action)
	if is_final:
		answer = extract_answer(action)
	else:
		memory += final_thought + final_action + final_result
return answer
```





### Meta-Analysis of Probabilistic ReAct

Currently each iteration of Probabilistic ReAct is making $n$ calls to an LLM API, one to get action specification, one to get multiple fill-ins and one for judging the results. That can be too many LLM calls leading to more expense on tokens; but it can potentially help in keeping the context length low.

Anyways, my hypothesis for Probabilistic ReAct is that since at every step, it chooses the best-of-n and is simulating a beam search, its performance should be strictly better than naive ReAct.

### Implementation of Probabilistic ReAct

The approach is current implemented as [TraceletCodeAgent](src/smolagents/tracelet_agent.py) and I describe the three LLM calls below:

1. **Sampling Specification**: We ask the LLM to output a templated action (in direct_prompt) or ask it to output a normal action and then we templatize it (in post_process)
2. **Filling in**: We ask the LLM to generate fillins in a structured format
3. **LLM as a judge**: We pass only the initial task, the actions and the corresponding results to an LLM to pick the best one

Results below: direct_prompt for all four models (Qwen3.7-Plus's rerun is still partial at 37/165, so its row is scored on those 37 questions), then the partial post_process runs.

In [6]:
NAIVE_DIRS = {
    "gpt-4o": MODEL_DIRS["gpt-4o"],
    "gpt-5.4-mini": MODEL_DIRS["gpt-5.4-mini"],
    "Qwen3.5-9B": MODEL_DIRS["Qwen3.5-9B"],
    "Qwen3.7-Plus": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.7-Plus_False",
}
TRACELET_DIRECT_DIRS = {
    "gpt-4o": BASELINE_DIR / "tracelet_direct_react_gpt-4o",
    "gpt-5.4-mini": BASELINE_DIR / "tracelet_direct_react_gpt-5.4-mini",
    "Qwen3.5-9B": BASELINE_DIR / "tracelet_direct_react_Qwen" / "Qwen3.5-9B",
    "Qwen3.7-Plus": BASELINE_DIR / "tracelet_direct_react_Qwen" / "Qwen3.7-Plus",
}

# Unlike summarize_results (which reads every *.pkl in a dir), this reads a fixed range of
# indices so two runs at different stages of completion can be compared on the exact same
# subset of questions -- Qwen3.7-Plus's rerun is only at 37/165, so its row compares against
# naiveReAct on those same first 37 questions while the other three compare on all 165.
def summarize_first_n(pkl_dir: Path, n: int) -> dict:
    records = []
    for i in range(n):
        with open(pkl_dir / f"{i}.pkl", "rb") as f:
            records.append(pickle.load(f))
    correct = sum(1 for r in records if r.get("is_correct"))
    return {
        "n": n,
        "correct": correct,
        "accuracy": correct / n,
        "avg_steps": sum(r.get("num_steps", 0) for r in records) / n,
        "avg_tokens": sum(r["token_counts"].get("total_tokens", 0) for r in records) / n,
        "max_steps_hits": sum(1 for r in records if r.get("num_steps", 0) >= 51),
    }


def build_table(dirs: dict, n_common_by_model: dict) -> pd.DataFrame:
    rows = []
    for model_name, pkl_dir in dirs.items():
        stats = summarize_first_n(pkl_dir, n_common_by_model[model_name])
        rows.append({
            "Model": model_name,
            "Accuracy": f"{stats['correct']}/{stats['n']} = {stats['accuracy']:.1%}",
            "Avg Steps": f"{stats['avg_steps']:.1f}",
            "Avg Tokens": f"{stats['avg_tokens']:,.0f}",
            "Hit max_steps (51)": stats["max_steps_hits"],
        })
    return pd.DataFrame(rows).set_index("Model")


# Same subset of questions for both tables per model, in case the Tracelet sweep for a
# given model isn't fully caught up yet.
n_common_by_model = {m: len(list(d.glob("*.pkl"))) for m, d in TRACELET_DIRECT_DIRS.items()}

# The n_samples=1 ablation is at a different stage of completion per model, so it gets its own
# subset sizes and its own matched naiveReAct table rather than reusing n_common_by_model above.
TRACELET_DIRECT_N1_DIRS = {
    "gpt-4o": BASELINE_DIR / "tracelet_direct_n1_react_gpt-4o",
    "gpt-5.4-mini": BASELINE_DIR / "tracelet_direct_n1_react_gpt-5.4-mini",
    "Qwen3.5-9B": BASELINE_DIR / "tracelet_direct_n1_react_Qwen" / "Qwen3.5-9B",
    "Qwen3.7-Plus": BASELINE_DIR / "tracelet_direct_n1_react_Qwen" / "Qwen3.7-Plus",
}
n1_common_by_model = {m: len(list(d.glob("*.pkl"))) for m, d in TRACELET_DIRECT_N1_DIRS.items()}

print("=== naiveReAct (matched to the n_samples=1 subsets) ===")
display(build_table(NAIVE_DIRS, n1_common_by_model))

print("=== Tracelet (direct_prompt, n_samples=1) ===")
display(build_table(TRACELET_DIRECT_N1_DIRS, n1_common_by_model))

# post_process coverage is partial and heterogeneous -- gpt-4o's sweep stopped at 12 questions, and the
# only other post_process run is Qwen3.7-Plus at n_samples=1 -- so each is paired with its own naiveReAct
# baseline over its own matched subset rather than dropped into the table above.
TRACELET_POSTPROCESS_RUNS = {
    "gpt-4o (n=3)": (BASELINE_DIR / "tracelet_react_gpt-4o", MODEL_DIRS["gpt-4o"]),
    "Qwen3.7-Plus (n=1)": (
        BASELINE_DIR / "tracelet_postprocess_n1_react_Qwen" / "Qwen3.7-Plus",
        BASELINE_DIR / "naive_react_Qwen" / "Qwen3.7-Plus_False",
    ),
}

postprocess_rows = []
for label, (pp_dir, naive_dir) in TRACELET_POSTPROCESS_RUNS.items():
    n = len(list(pp_dir.glob("*.pkl")))
    pp, naive = summarize_first_n(pp_dir, n), summarize_first_n(naive_dir, n)
    postprocess_rows.append({
        "Config": label,
        "Questions": n,
        "naive Accuracy": f"{naive['correct']}/{n} = {naive['accuracy']:.1%}",
        "post_process Accuracy": f"{pp['correct']}/{n} = {pp['accuracy']:.1%}",
        "naive Steps": f"{naive['avg_steps']:.1f}",
        "post_process Steps": f"{pp['avg_steps']:.1f}",
        "naive Tokens": f"{naive['avg_tokens']:,.0f}",
        "post_process Tokens": f"{pp['avg_tokens']:,.0f}",
        "post_process hit max_steps": pp["max_steps_hits"],
    })

print("\n=== Tracelet (post_process) vs naiveReAct, each on its own matched subset ===")
display(pd.DataFrame(postprocess_rows).set_index("Config"))


=== naiveReAct (matched to the n_samples=1 subsets) ===


,Accuracy,Avg Steps,Avg Tokens,Hit max_steps (51)
Model,,,,
gpt-4o,43/165 = 26.1%,10.8,"136,605",5
gpt-5.4-mini,54/165 = 32.7%,5.4,"37,291",1
Qwen3.5-9B,35/64 = 54.7%,18.1,"409,701",6
Qwen3.7-Plus,28/50 = 56.0%,20.1,"466,413",6


=== Tracelet (direct_prompt, n_samples=1) ===


,Accuracy,Avg Steps,Avg Tokens,Hit max_steps (51)
Model,,,,
gpt-4o,42/165 = 25.5%,18.2,"369,365",17
gpt-5.4-mini,47/165 = 28.5%,13.8,"278,993",14
Qwen3.5-9B,22/64 = 34.4%,29.5,"457,844",27
Qwen3.7-Plus,13/50 = 26.0%,34.4,"1,088,872",27



=== Tracelet (post_process) vs naiveReAct, each on its own matched subset ===


,Questions,naive Accuracy,post_process Accuracy,naive Steps,post_process Steps,naive Tokens,post_process Tokens,post_process hit max_steps
Config,,,,,,,,
gpt-4o (n=3),12,3/12 = 25.0%,2/12 = 16.7%,14.1,20.2,"204,702","405,611",2
Qwen3.7-Plus (n=1),50,28/50 = 56.0%,8/50 = 16.0%,20.1,40.0,"466,413","925,362",30


Ok, so from the results above, it is very clear that Tracelet (with n=1) does not perform as well as naiveReAct which is concerning since it should have performed competitively. It is just naiveReAct with some additional bells and whistles. I spent few days concerned about bells and whistles being implemented to cover as many failure cases as possible.
- made sure memory was updated appropriately
- fillins had fallbacks
- there were sentinels in the produced code

But it also seems that since we never changed the `system prompt`, the model doesn't know what is happening. So yes, in the next step, let's write a suitable system prompt and actually change TraceletReAct into something where the model knows what is happening.

### Probabilistic ReAct v3: protocol prompt + snapshot commit

The accuracy drop diagnosed above was fixed by three changes (2026-08-24/25):

1. **Protocol system prompt** (`src/smolagents/prompts/tracelet_agent.yaml`, passed via `prompt_templates=`): teaches the template/fill-in cycle with rewritten few-shot examples (sentinelized tool calls + `Fill-in:` blocks), plus a verify-first rule that stopped gpt-5.4-mini from answering from priors in one step (28% → 42% on the first 50).
2. **Memory replays the protocol**: `model_output` records the skeleton + chosen `Fill-in:` lines instead of the substituted code, so the model's own history keeps demonstrating the format (without this, sentinel emission decays over the trajectory).
3. **Snapshot commit**: the judged winner is committed from its trial's saved executor state instead of being re-executed. The old double execution corrupted stateful-tool observations (`page_down` advanced two pages per step, so the model read every other page).

The table below compares naiveReAct against this v3 configuration (`direct_prompt`, `n_samples=1` -- the ablation, so parity with naive is the expected result; best-of-n gains would come from n>1, not yet rerun). Each pair is scored on the same first-N questions, where N is however many the (possibly still-running) v3 sweep has completed for that model.

In [12]:
# Reuses BASELINE_DIR, summarize_first_n, pd, and display from the cells above.

TRACELET_V3_DIRS = {
    "gpt-4o": BASELINE_DIR / "tracelet_direct_n1_tp_v3_react_gpt-4o",
    "gpt-5.4-mini": BASELINE_DIR / "tracelet_direct_n1_tp_v3_react_gpt-5.4-mini",
    "Qwen3.5-9B": BASELINE_DIR / "tracelet_direct_n1_tp_v3_react_Qwen" / "Qwen3.5-9B",
    "Qwen3.7-Plus": BASELINE_DIR / "tracelet_direct_n1_tp_v3_react_Qwen" / "Qwen3.7-Plus",
}
V3_NAIVE_DIRS = {
    "gpt-4o": BASELINE_DIR / "naive_react_gpt-4o_False",
    "gpt-5.4-mini": BASELINE_DIR / "naive_react_gpt-5.4-mini_False",
    "Qwen3.5-9B": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.5-9B_False",
    "Qwen3.7-Plus": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.7-Plus_False",
}

v3_rows = []
for model_name, v3_dir in TRACELET_V3_DIRS.items():
    naive_dir = V3_NAIVE_DIRS[model_name]
    # Matched first-N subset per model: N = whatever the (possibly still-running) v3 sweep has finished.
    n = min(len(list(v3_dir.glob("*.pkl"))), len(list(naive_dir.glob("*.pkl")))) if v3_dir.exists() else 0
    if n == 0:
        v3_rows.append({"Model": model_name, "Questions": 0, "naive Accuracy": "no data", "prob Accuracy": "no data"})
        continue
    naive, prob = summarize_first_n(naive_dir, n), summarize_first_n(v3_dir, n)
    v3_rows.append({
        "Model": model_name,
        "Questions": n,
        "naive Accuracy": f"{naive['correct']}/{n} = {naive['accuracy']:.1%}",
        "prob Accuracy": f"{prob['correct']}/{n} = {prob['accuracy']:.1%}",
        "naive Steps": f"{naive['avg_steps']:.1f}",
        "prob Steps": f"{prob['avg_steps']:.1f}",
        "naive Tokens": f"{naive['avg_tokens']:,.0f}",
        "prob Tokens": f"{prob['avg_tokens']:,.0f}",
        # How much more the probabilistic version spends per question than naive.
        "Token Increase": (
            f"{prob['avg_tokens'] / naive['avg_tokens']:.1f}x (+{100 * (prob['avg_tokens'] / naive['avg_tokens'] - 1):.0f}%)"
            if naive["avg_tokens"]
            else "-"
        ),
    })

display(pd.DataFrame(v3_rows).set_index("Model"))

=== naiveReAct vs Probabilistic ReAct (Tracelet direct_prompt, n=1, v3 prompt), matched subsets ===


,Questions,naive Accuracy,prob Accuracy,naive Steps,prob Steps,naive Tokens,prob Tokens,Token Increase
Model,,,,,,,,
gpt-4o,165,43/165 = 26.1%,38/165 = 23.0%,10.8,12.9,"136,605","415,569",3.0x (+204%)
gpt-5.4-mini,165,54/165 = 32.7%,50/165 = 30.3%,5.4,9.5,"37,291","329,657",8.8x (+784%)
Qwen3.5-9B,165,83/165 = 50.3%,68/165 = 41.2%,18.9,21.5,"467,797","916,970",2.0x (+96%)
Qwen3.7-Plus,132,82/132 = 62.1%,70/132 = 53.0%,19.0,20.8,"456,244","1,058,984",2.3x (+132%)


Clearly there is a significant drop in accuracy for TraceletReAct. There are few possible reasons for this:
1. Making the model output the template is decreasing its performance
2. Model is failing to output the fillins for all sentinels in a template

To isolate these effects, I am running experiments with n=1 and the accuracy should be strictly less than n=3

Human Notes:
- TODO: extracting probability for the actions (you can get from OpenAI or TogetherAI)
- formal questions about how they are helping in modeling
- reflexion - more concretely, the model generates a unit test
    - do we want to output observes and use them to guide feedback
- Question: which thing need probability - action output, feedback should be probability
    - how do you interpret it
    How often is the solution ranked one
    How often are we consistent - is the judge sucks
        Formally, we are saying that the probabilty of generation should correlate with probability of judging the option to be the most correct
        Light version of verification
        Verification of language model programming
    The judge sucks - we can signal that 

- Added structure to have probabilistic structure to diagnose models
    - Is the first solution better than the second solution
    - If the model is good, than it should rank things according to score
    - Language model judge should point to the most correct solutions - verifying the judge
    - We can measure stuff and these are the things we can measure
        - We don't lose performance over Vanilla ReAct
        - We improve performance
        - We can also optimize the programs to increase performance and efficiency


By the end of this month,
- have a prototype of probabilistic ReAct
- empirical results of GAIA and maybe Fermi, and maybe some other dataset
- plugin to the language Kyle is building 
    - pluginto tensors and circuits
- traces, analysis
        - 

At full scale (165/165 for all three models), the picture doesn't support the "best-of-n should be strictly better" hypothesis above: Tracelet (direct_prompt) comes out roughly flat or behind naiveReAct on accuracy for every model, at meaningfully higher token cost (~2.5x for gpt-4o, ~8x for gpt-5.4-mini, roughly even for Qwen3.5-9B) and more steps across the board.

Before reading too much into the accuracy gaps, though: these are single runs per condition, and this exact naiveReAct setup is already documented above as varying significantly run to run. Rough two-proportion check (n=165 each): gpt-4o's gap is ~1 question, indistinguishable from noise. gpt-5.4-mini's ~6.7pp gap is ~1.3 standard errors, still plausibly noise. Qwen3.5-9B's ~12.7pp gap is ~2.3 standard errors -- looks like a real signal, but it's still one run against one run, so treat it as suggestive, not conclusive, without repeats.

Manually inspecting several trajectories also surfaced a real bug worth fixing before trusting these numbers much further: when the model's fill-in response for a sentinel doesn't match the expected `<sentinel>: <value>` line format exactly (e.g. wrapped in backticks or a leading bullet), `_parse_fillin_lines` silently drops that line and the sentinel is left unsubstituted in the code. If the judge then scores all `n` candidates identically (e.g. because none made real progress), `_pick_best`'s tie-break defaults to candidate 0 regardless of whether it's actually viable -- so a candidate with a literally unresolved sentinel variable can still get committed and executed, surfacing as `InterpreterError: The variable ARG0 is not defined`. Seen this twice independently across the trajectories inspected so far (not a fluke) -- not yet fixed. Fix: make `_pick_best` deprioritize/exclude candidates whose trial observation started with `"Error:"` instead of treating them as tie-eligible.

## Related Work

[Self Compacting Language Model Agents](https://arxiv.org/pdf/2604.17290) prompts a language model to fill up a rubric and decided to compress or not every N steps in order to deal with context rot and ever increasing context. 
- Can be a good place to check for alternate datasets

[PPDL](https://arxiv.org/abs/2608.05234) has built a probabilistic programming language for language model prompting. It provides a framework where people can provide the prompt for the language model and provide constraints (hard syntactic constraints or soft LLM-as-a-judge constraints) using the factor construct. Using this framework they show how IS and SMC can be used as inference algorithms in this flow. Very similar to [Encompass](https://arxiv.org/pdf/2512.03571) but with probabilities. 


Fermi is less markovian where it can be harder to do better than naive truncation
Establish baselines to reduce token budget effectively
Open weight models are performant and still have room to improve.
s